# Laya × Integrated Memory V2.2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V22_Colab.ipynb)

Corrected bidirectional Integrated Memory V2.2 adaptation for ModernBERT. Pretrained `Wqkv`/`Wo` stay frozen; full-attention blocks are converted first with a learned positive-kernel memory, plus CeNN-style local and direct-value residual routes.

**Key correction.** Training now uses the repository's fixed held-out local probe and supervises both the teacher's **pre-`Wo` attention core** and final attention output. Q/K feature maps start in the same feature space instead of unrelated random ELU maps.

A proposed layer is still accepted **one layer at a time** only when the original strict local-fidelity and Laya decision-level gates pass. Rejected layers are restored automatically.


## 1. Setup
A T4/L4/A100 runtime is recommended. The notebook installs the latest Laya source plus this TinyCeNN-LM repository.


In [1]:
import os, sys, subprocess, pathlib, importlib
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Works in Colab, Kaggle, and ordinary Jupyter kernels.
if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REPO = WORK / "TinyCeNN-LM"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "pull", "-q"])

# IMPORTANT: install with this notebook kernel's Python, not a possibly different `pip` executable.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NandhaKishorM/laya.git", "datasets", "pandas", "pyarrow", "safetensors"])

# Editable-install fallback for notebook environments that cache import paths.
SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import compileall
LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
assert compileall.compile_dir(str(LAB_SRC), quiet=1), "Python syntax preflight failed in tinycenn_lm/laya_lab"

import tinycenn_lm
from tinycenn_lm.laya_lab import LayaLabConfig, run_experiment
import torch, json, pandas as pd
print("TinyCeNN-LM:", pathlib.Path(tinycenn_lm.__file__).resolve())
print("torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


TinyCeNN-LM: /content/TinyCeNN-LM/src/tinycenn_lm/__init__.py
torch: 2.11.0+cu128 GPU: Tesla T4


## 2. Experiment configuration
The validated high-LR run converged much faster than the previous schedule: layer 18 reached strict acceptance with `LR=0.01`, NMSE `0.116`, cosine `0.930`, teacher agreement `0.987`, and KL `0.00041`.

This notebook therefore uses `LR=0.01`, `feature_dim=128`, and a **1200-step maximum per layer with adaptive early stopping**. It automatically attempts **every full-attention ModernBERT block**, one at a time. Each candidate is retained only if the unchanged local- and decision-level gates pass; otherwise that block is restored to the original Laya attention.

Sliding-attention blocks remain native because Integrated Memory V2.2 currently implements a global bidirectional kernel memory rather than ModernBERT's query-local sliding window. Replacing those blocks with this kernel would change their topology rather than perform a faithful conversion.


In [6]:
MODEL_ID = "convaiinnovations/laya-typed-decisions" #@param ["convaiinnovations/laya-typed-decisions", "convaiinnovations/laya"]
MODE = "extended" #@param ["smoke", "balanced", "extended"]

# Optimized from the successful layer-18 run.
cfg = LayaLabConfig(
    architecture="integrated_memory_v22",
    model_id=MODEL_ID,
    mode=MODE,
    seed=2026,
    target_layers=None,
    target_all_full_attention=True,  # automatically convert every full-attention block
    feature_dim=128,                  # 256 effective positive features/head
    local_kernel=5,
    learning_rate=1e-2,               # validated fast-convergence LR
    weight_decay=1e-3,
    training_steps=2400,              # maximum; adaptive early-stop can finish sooner
    early_stop_local=False,
    # Keep strict acceptance gates unchanged.
    max_local_nmse=0.20,
    min_local_cosine=0.90,
    min_teacher_agreement=0.90,
    max_mean_kl=0.005,
    max_accuracy_drop=0.005,
    output_dir="/content/laya_tinycenn",
)
print(cfg)
print("Target policy: all full-attention ModernBERT blocks")
print("Effective feature dimension/head:", 2 * cfg.feature_dim)
print("Maximum training steps/layer:", cfg.training_steps)
print("Adaptive local early stop:", cfg.early_stop_local)
print("Transfer objective: 0.75*NMSE + 1.00*(1-cos) + 0.35*core_NMSE + 0.30*(1-core_cos)")


LayaLabConfig(architecture='integrated_memory_v22', model_id='convaiinnovations/laya-typed-decisions', mode='extended', seed=2026, feature_dim=128, memory_rank=32, local_kernel=5, pdelta_conv_kernel=4, pdelta_chunk_size=32, learning_rate=0.01, weight_decay=0.001, training_steps=2400, target_layers=None, target_all_full_attention=True, early_stop_local=False, min_teacher_agreement=0.9, max_mean_kl=0.005, max_accuracy_drop=0.005, max_local_nmse=0.2, min_local_cosine=0.9, output_dir='/content/laya_tinycenn')
Target policy: all full-attention ModernBERT blocks
Effective feature dimension/head: 256
Maximum training steps/layer: 2400
Adaptive local early stop: False
Transfer objective: 0.75*NMSE + 1.00*(1-cos) + 0.35*core_NMSE + 0.30*(1-core_cos)


## 3. Convert all full-attention transformer blocks with strict sequential acceptance
The runner discovers every `full_attention` ModernBERT layer at runtime. It starts with the empirically strongest transfer layers (18, then 21 when present) and then covers the remaining full-attention blocks. Accepted replacements stay active; rejected replacements are immediately restored.

The high-LR schedule uses a maximum of 1200 steps per layer. After step 400, training can stop early when two consecutive fixed-probe evaluations satisfy a stronger margin than the acceptance gate: NMSE ≤ 0.20, cosine ≥ 0.90, core NMSE ≤ 0.20, and core cosine ≥ 0.90.

Final acceptance remains unchanged:

- attention-output NMSE ≤ 0.30 and cosine ≥ 0.88;
- teacher top-1 agreement ≥ 0.95;
- teacher→student mean KL ≤ 0.05;
- held-out accuracy drop ≤ 0.02.

The final evaluation and fast evaluation use the **cumulative student containing every accepted replacement**.


In [7]:
teacher, student, report = run_experiment(cfg)


Loading Laya teacher: convaiinnovations/laya-typed-decisions


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Attention-transfer sequences: 5000 | train max length: 768
Teacher gate accuracy: 0.748
Candidate ModernBERT layers: [(18, 'full_attention'), (21, 'full_attention'), (6, 'full_attention'), (12, 'full_attention'), (15, 'full_attention'), (9, 'full_attention'), (3, 'full_attention'), (24, 'full_attention'), (0, 'full_attention'), (27, 'full_attention')]

=== integrated_memory_v22: layer 18 (full_attention) ===
layer=18 step=0001/2400 train_nmse=0.9595 train_cos=0.3211 val_nmse=0.9860 val_cos=0.3022 val_core_nmse=0.7977
layer=18 step=0200/2400 train_nmse=0.2916 train_cos=0.8358 val_nmse=0.2067 val_cos=0.8762 val_core_nmse=0.1662
layer=18 step=0400/2400 train_nmse=0.1543 train_cos=0.8952 val_nmse=0.1805 val_cos=0.8925 val_core_nmse=0.1427
layer=18 step=0600/2400 train_nmse=0.1972 train_cos=0.8892 val_nmse=0.1603 val_cos=0.9051 val_core_nmse=0.1293
layer=18 step=0800/2400 train_nmse=0.1461 train_cos=0.9164 val_nmse=0.1521 val_cos=0.9093 val_core_nmse=0.1220
layer=18 step=1000/2400 train_nms

## 4. Laya-native result summary
The final report emphasizes what matters for Laya: typed decision accuracy, soft-label agreement, Brier/ECE, score MAE, teacher agreement, and request latency.


In [8]:
converted = bool(report.get("accepted_layers"))
student_label = report["architecture"] if converted else report["architecture"] + " (no layer accepted; teacher restored)"
summary = pd.DataFrame([
    {"model":"Laya teacher", **{k: report["teacher_final"].get(k) for k in ["accuracy","soft_accuracy","brier","brier_vs_soft","ece","score_mae","ms_per_case"]}},
    {"model":student_label, **{k: report["student_final"].get(k) for k in ["accuracy","soft_accuracy","brier","brier_vs_soft","ece","score_mae","ms_per_case"]},
     "teacher_agreement": report["student_final"].get("teacher_agreement"),
     "teacher_KL": report["student_final"].get("mean_teacher_kl")},
])
display(summary)
print("Conversion succeeded:", converted)
print("Accepted attention layers:", report["accepted_layers"])
print("Target policy:", report.get("target_policy"))
print("Accepted/attempted:", f'{len(report["accepted_layers"])}/{len(report["candidate_layers"])}')
print("Training steps/candidate:", report.get("training_steps_per_candidate"))
print("Replacement trainable parameters:", f'{report["replacement_trainable_parameters"]:,}')
print("Latency:", json.dumps(report["latency"], indent=2))

for primitive, metrics in report["student_final"]["by_type"].items():
    print(primitive, metrics)


,model,accuracy,soft_accuracy,brier,brier_vs_soft,ece,score_mae,ms_per_case,teacher_agreement,teacher_KL
0,Laya teacher,0.7665,0.470659,0.399592,0.061466,0.213739,0.24241,136.859015,NaN,NaN
1,integrated_memory_v22,0.7545,0.463569,0.412247,0.065817,0.213918,0.26204,202.161627,0.939,0.004549


Conversion succeeded: True
Accepted attention layers: [6, 15, 18, 21, 24, 27]
Target policy: all_full_attention
Accepted/attempted: 6/10
Training steps/candidate: 2400
Replacement trainable parameters: 1,604,064
Latency: {
  "teacher": {
    "median_ms": 40.480471499904525,
    "p95_ms": 43.83465030041407,
    "mean_ms": 40.78821954990417,
    "repeats": 20
  },
  "student": {
    "median_ms": 48.07170899994162,
    "p95_ms": 52.427622649929624,
    "mean_ms": 48.65515750002487,
    "repeats": 20
  }
}
choice {'n': 600, 'accuracy': 0.7333333333333333, 'brier': 0.4707473279571622, 'nll': 0.8752028307448747, 'ece': 0.2634581241394917, 'soft_accuracy': 0.3938432036816146, 'brier_vs_soft': 0.08474452385783678, 'score_mae': None}
noul {'n': 600, 'accuracy': 0.8433333333333334, 'brier': 0.29242947049999996, 'nll': 0.46286748966029007, 'ece': 0.18331450000000005, 'soft_accuracy': 0.6155785959936667, 'brier_vs_soft': 0.05133579595725333, 'score_mae': None}
score {'n': 800, 'accuracy': 0.70375,

In [9]:
# === FAST EVAL: 50 held-out typed-decision cases ===
from tinycenn_lm.laya_lab.data import _load_typed_split, dataset_cases
from tinycenn_lm.laya_lab.evaluate import evaluate_agent

FAST_EVAL_CASES = 50 #@param {type:"integer"}
fast_ds = _load_typed_split("test")
fast_cases = dataset_cases(fast_ds, FAST_EVAL_CASES)

teacher_fast = evaluate_agent(teacher, fast_cases, label="teacher_fast")
student_fast = evaluate_agent(
    student,
    fast_cases,
    teacher_agent=teacher,
    label="integrated_memory_v22_fast",
)

fast_table = pd.DataFrame([
    {
        "model": "Laya teacher",
        "accuracy": teacher_fast.get("accuracy"),
        "soft_accuracy": teacher_fast.get("soft_accuracy"),
        "nll": teacher_fast.get("nll"),
        "brier": teacher_fast.get("brier"),
        "ece": teacher_fast.get("ece"),
        "ms_per_case": teacher_fast.get("ms_per_case"),
    },
    {
        "model": "Integrated Memory V2.2",
        "accuracy": student_fast.get("accuracy"),
        "soft_accuracy": student_fast.get("soft_accuracy"),
        "nll": student_fast.get("nll"),
        "brier": student_fast.get("brier"),
        "ece": student_fast.get("ece"),
        "ms_per_case": student_fast.get("ms_per_case"),
        "teacher_agreement": student_fast.get("teacher_agreement"),
        "teacher_KL": student_fast.get("mean_teacher_kl"),
    },
])
display(fast_table)

teacher_ms = float(teacher_fast.get("ms_per_case") or 0.0)
student_ms = float(student_fast.get("ms_per_case") or 0.0)
speed_ratio = teacher_ms / student_ms if student_ms > 0 else float("nan")
print("Accepted layers      :", report.get("accepted_layers", []))
print("Teacher agreement    :", round(float(student_fast.get("teacher_agreement", 0.0)), 4))
print("Mean teacher KL      :", round(float(student_fast.get("mean_teacher_kl", 0.0)), 6))
print("Teacher ms/case      :", round(teacher_ms, 3))
print("Student ms/case      :", round(student_ms, 3))
print("Teacher/student ratio:", round(speed_ratio, 3), "x")
if not report.get("accepted_layers"):
    print("NOTE: no layer was accepted, so the student is the restored teacher; this is not a converted-model speed result.")


,model,accuracy,soft_accuracy,nll,brier,ece,ms_per_case,teacher_agreement,teacher_KL
0,Laya teacher,0.736,0.415219,0.792255,0.455453,0.230092,84.402878,NaN,NaN
1,Integrated Memory V2.2,0.724,0.410689,0.806204,0.465323,0.231020,102.913955,0.956,0.001709


Accepted layers      : [6, 15, 18, 21, 24, 27]
Teacher agreement    : 0.956
Mean teacher KL      : 0.001709
Teacher ms/case      : 84.403
Student ms/case      : 102.914
Teacher/student ratio: 0.82 x


## 5. Re-run your Laya example on the adapted model
The same 'Router' API is preserved. We attach the already-loaded adapted Agent so there is no second model copy. The architecture here replaces the **English Laya** encoder only; Laya's multilingual route can remain the original multilingual checkpoint.


In [10]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"]
    },
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"},
    "refund_requested": {"type": "noul", "instructions": "Does the user explicitly request a refund?"}
}

route_name = "typed-decisions" if "typed-decisions" in MODEL_ID else "english"
router = Router(preload=False)
router.attach(route_name, student)
res = router.predict(state, questions, model=route_name)

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw demo comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


Department       : billing
Urgency score    : 1.6032
Churn risk       : 0.6471
Refund requested : 0.6894
Routing          : typed-decisions

Teacher/student raw demo comparison:
{
  "teacher": {
    "model": "laya-rl-agent",
    "answers": {
      "department": {
        "type": "choice",
        "choice": "billing",
        "probabilities": {
          "billing": 0.8037,
          "technical": 0.0677,
          "sales": 0.0608,
          "other": 0.0679
        },
        "confidence": 0.4873,
        "action": {
          "act_probability": 1.0
        }
      },
      "urgency": {
        "type": "score",
        "score": 1.6379,
        "legend": {
          "0": "not urgent",
          "1": "soon",
          "2": "critical deadline or blocking issue"
        },
        "probabilities": {
          "0": 0.044,
          "1": 0.2742,
          "2": 0.6819
        },
        "confidence": 0.3144,
        "action": {
          "act_probability": 1.0
        }
      },
      "churn_ris

## 6. Inspect layer-by-layer acceptance
This table reports projected-output fidelity, pre-`Wo` core fidelity, and decision-level gates. A falling training loss alone is not treated as successful conversion.


In [11]:
rows=[]
for h in report["history"]:
    rows.append({
        "layer":h["layer"], "attention_type":h["attention_type"], "accepted":h["accepted"],
        "nmse":h["local"]["nmse"], "cosine":h["local"]["cosine"],
        "core_nmse":h["local"].get("core_nmse"), "core_cosine":h["local"].get("core_cosine"),
        "selected_step":h["local"].get("step"), "stopped_early":h["local"].get("stopped_early"), "selection":h["local"].get("selection"),
        "teacher_agreement":h["gate"].get("teacher_agreement"),
        "mean_teacher_kl":h["gate"].get("mean_teacher_kl"),
        "accuracy":h["gate"].get("accuracy"), "accuracy_drop":h["accuracy_drop"],
    })
display(pd.DataFrame(rows))


,layer,attention_type,accepted,nmse,cosine,core_nmse,core_cosine,selected_step,stopped_early,selection,teacher_agreement,mean_teacher_kl,accuracy,accuracy_drop
0,18,full_attention,True,0.116088,0.929522,0.095442,0.950710,2400,False,fixed_local_probe,0.987,0.000406,0.747,0.001
1,21,full_attention,True,0.146958,0.919113,0.123966,0.934615,2400,False,fixed_local_probe,0.991,0.000496,0.747,0.001
2,6,full_attention,True,0.131972,0.929439,0.141161,0.924015,2400,False,fixed_local_probe,0.983,0.001060,0.753,-0.005
3,12,full_attention,False,0.121567,0.939693,0.109139,0.946584,2400,False,fixed_local_probe,0.943,0.007862,0.739,0.009
4,15,full_attention,True,0.144973,0.921647,0.100668,0.947652,2400,False,fixed_local_probe,0.950,0.004937,0.744,0.004
5,9,full_attention,False,0.163995,0.916693,0.100150,0.949234,2400,False,fixed_local_probe,0.948,0.005687,0.745,0.003
6,3,full_attention,False,0.086736,0.950428,0.085577,0.951321,2400,False,fixed_local_probe,0.943,0.005318,0.741,0.007
7,24,full_attention,True,0.092654,0.949679,0.102077,0.942230,2400,False,fixed_local_probe,0.949,0.004962,0.747,0.001
8,0,full_attention,False,0.065015,0.963063,0.079209,0.953333,2400,False,fixed_local_probe,0.940,0.005053,0.743,0.005
9,27,full_attention,True,0.075060,0.963644,0.056662,0.972364,2400,False,fixed_local_probe,0.949,0.004944,0.747,0.001


## 7. Saved outputs
The notebook writes an adapter-only PyTorch checkpoint plus a JSON report under '/content/laya_tinycenn/<architecture>/'. The report contains the acceptance history, official Laya-style evaluation, latency measurements, and the duplicate-charge demo.


In [12]:
from pathlib import Path
out = Path(cfg.output_dir) / cfg.architecture
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print((out / "report.json").read_text()[:4000])


Adapter: /content/laya_tinycenn/integrated_memory_v22/adapter.pt
Report : /content/laya_tinycenn/integrated_memory_v22/report.json
{
  "architecture": "integrated_memory_v22",
  "model_id": "convaiinnovations/laya-typed-decisions",
  "mode": "extended",
  "candidate_layers": [
    18,
    21,
    6,
    12,
    15,
    9,
    3,
    24,
    0,
    27
  ],
  "target_policy": "all_full_attention",
  "accepted_layers": [
    6,
    15,
    18,
    21,
    24,
    27
  ],
  "replacement_trainable_parameters": 1604064,
  "training_steps_per_candidate": 2400,
  "history": [
    {
      "layer": 18,
      "attention_type": "full_attention",
      "local": {
        "nmse": 0.11608751863241196,
        "cosine": 0.9295223951339722,
        "core_nmse": 0.09544196724891663,
        "core_cosine": 0.9507097005844116,
        "step": 2400,
        "selection": "fixed_local_probe",
        "stopped_early": false,
        "max_steps": 2400
      },
      "gate": {
        "n": 1000,
        "accura

In [13]:
# === SAVE ADAPTED MODEL TO HUGGING FACE HUB ===
# Uploads a reloadable TinyCeNN adapter package: base Laya + adapter.pt + loader.py.
from pathlib import Path
import json, os, shutil
from huggingface_hub import HfApi, login

HF_REPO_ID = "vtava/Laya-Integrated-Memory-V22" #@param {type:"string"}
HF_PRIVATE = False #@param {type:"boolean"}

accepted_layers = report.get("accepted_layers", [])
if not accepted_layers:
    raise RuntimeError(
        "No Integrated Memory layer passed the strict gates. "
        "HF upload is intentionally blocked so an unchanged teacher is not published as a converted model."
    )

out = Path(cfg.output_dir) / cfg.architecture
adapter_path = out / "adapter.pt"
report_path = out / "report.json"
if not adapter_path.exists() or not report_path.exists():
    raise FileNotFoundError("Run the training/evaluation cells first; adapter.pt/report.json are missing.")

export_dir = out / "hf_export"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(adapter_path, export_dir / "adapter.pt")
shutil.copy2(report_path, export_dir / "report.json")

meta = {
    "format": "tinycenn-laya-attention-lab-v1",
    "architecture": cfg.architecture,
    "base_model": cfg.model_id,
    "accepted_layers": accepted_layers,
    "candidate_layers": report.get("candidate_layers", []),
    "target_policy": report.get("target_policy"),
    "learning_rate": cfg.learning_rate,
    "training_steps_max": cfg.training_steps,
    "early_stop_local": cfg.early_stop_local,
    "feature_dim": cfg.feature_dim,
    "local_kernel": cfg.local_kernel,
    "tinycenn_repo": "https://github.com/vtavakkoli/TinyCeNN-LM",
    "notebook": "notebooks/Laya_Integrated_Memory_V22_Colab.ipynb",
}
(export_dir / "model_meta.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
)

loader_source = '''from pathlib import Path
import torch
from huggingface_hub import snapshot_download
import laya
from tinycenn_lm.laya_lab.core import LayaLabConfig
from tinycenn_lm.laya_lab.factory import make_replacement


def load_model(repo_id, device=None, token=None):
    root = Path(snapshot_download(
        repo_id,
        repo_type="model",
        token=token,
        allow_patterns=["adapter.pt", "report.json", "model_meta.json"],
    ))
    payload = torch.load(root / "adapter.pt", map_location="cpu", weights_only=False)
    cfg = LayaLabConfig(**payload["lab_config"])
    agent = laya.load(cfg.model_id, device=device, token=token)

    for layer_id, spec in payload["adapters"].items():
        idx = int(layer_id)
        layer = agent.model.encoder.layers[idx]
        replacement = make_replacement(layer.attn, cfg)
        replacement.load_state_dict(spec["state_dict"], strict=True)
        replacement.to(agent.device).eval().requires_grad_(False)
        layer.attn = replacement

    agent.model.eval()
    return agent
'''
(export_dir / "loader.py").write_text(loader_source, encoding="utf-8")

(export_dir / "requirements.txt").write_text(
    "git+https://github.com/NandhaKishorM/laya.git\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git\n"
    "huggingface_hub\n",
    encoding="utf-8",
)

readme = f'''---
library_name: pytorch
base_model: {cfg.model_id}
pipeline_tag: text-classification
tags:
- laya
- modernbert
- tinycenn
- linear-attention
- integrated-memory
---

# Laya Integrated Memory V2.2

TinyCeNN Integrated Memory V2.2 attention-conversion adapter for **{cfg.model_id}**.

- Accepted replacement layers: **{accepted_layers}**
- Attempted full-attention layers: **{report.get('candidate_layers', [])}**
- Transfer LR: **{cfg.learning_rate}**
- Max steps/layer: **{cfg.training_steps}**
- Adaptive early stop: **{cfg.early_stop_local}**
- Architecture: **{cfg.architecture}**
- Feature dimension: **{cfg.feature_dim}** (positive feature map expands this internally)
- Strict sequential acceptance was used; rejected layers were restored.

## Load

```python
from loader import load_model
agent = load_model("{HF_REPO_ID}", device="cuda")
```

The Hub repository stores the TinyCeNN adapter rather than duplicating the full base Laya checkpoint. `loader.py` downloads the declared base model and installs the accepted custom attention layers from `adapter.pt`.
'''
(export_dir / "README.md").write_text(readme, encoding="utf-8")

# Optional Colab secret: add HF_TOKEN under the key icon. Otherwise login() opens the HF auth flow.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    login()

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=HF_PRIVATE,
    exist_ok=True,
)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(export_dir),
    commit_message=(
        "Upload Laya Integrated Memory V2.2 adapter; "
        f"accepted layers={accepted_layers}"
    ),
)
print("Uploaded:", f"https://huggingface.co/{HF_REPO_ID}")
print("Package :", export_dir)


Uploaded: https://huggingface.co/vtava/Laya-Integrated-Memory-V22
Package : /content/laya_tinycenn/integrated_memory_v22/hf_export
